In [1]:
import pandas as pd
import numpy as np
import joblib

from surprise import SVD, Dataset, Reader

In [5]:
# Load training data
training_data = pd.read_csv("../data/training_data.csv")

# Load MovieLens links
links = pd.read_csv("../data/ml-32m/links.csv")

# Merge to add tmdbId
training_data = training_data.merge(
    links[["movieId", "tmdbId"]],
    on="movieId",
    how="left"
)

# Select CF features
cf_train = training_data[
    ["userId", "tmdbId", "rating"]
].dropna().copy()

# Convert tmdbId to integer
cf_train["tmdbId"] = cf_train["tmdbId"].astype(int)

print("Training shape:", cf_train.shape)
print("Unique users:", cf_train["userId"].nunique())
print("Unique movies:", cf_train["tmdbId"].nunique())

Training shape: (210831, 3)
Unique users: 1000
Unique movies: 12248


In [6]:
reader = Reader(rating_scale=(0.5, 5.0))

data = Dataset.load_from_df(
    cf_train,
    reader
)

trainset = data.build_full_trainset()

In [7]:
cf_model = SVD(
    n_factors=50,
    n_epochs=20,
    lr_all=0.005,
    reg_all=0.02,
    random_state=42
)

cf_model.fit(trainset)

In [8]:
joblib.dump(
    cf_model,
    "../models/collaborative_filtering.pkl"
)

['../models/collaborative_filtering.pkl']